In [1]:
import numpy as np
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import classification_report

from dotenv import load_dotenv
import os

from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate

In [2]:
load_dotenv('../.env')
print(os.environ['BASEURL'])

https://api.copilot.vk.team/v1


In [3]:
NUM_LABELS = 3  # 0=negative, 1=neutral, 2=positive

In [4]:
ds = load_dataset('./../datasets/reviews_cleaned')

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 119048
    })
})

In [6]:
set(ds['train']['language'])

{'kazakh', 'other', 'russian'}

In [7]:
ds_russian = ds.filter(lambda x: x['language'] in ('russian'))

In [8]:
def is_valid(example):
    t = example["combined_text"]
    if t is None:
        return False
    if isinstance(t, float) and np.isnan(t):
        return False
    if isinstance(t, str) and t.strip() == "":
        return False
    return True

ds_russian = ds_russian.filter(is_valid)

In [9]:
ds_russian

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 111667
    })
})

In [10]:
def calc_sentinent(row):
    sentiment = 1
    if row['rating'] > 3.0:
        sentiment = 2
    elif row['rating'] < 3.0:
        sentiment = 0
    else:
        sentiment = 1
    return {'label': sentiment}

In [11]:
ds_russian = ds_russian.map(calc_sentinent)

In [12]:
ds_russian['train']

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label'],
    num_rows: 111667
})

In [13]:
num_negative = ds_russian['train'].filter(lambda row: row['label'] == 0).num_rows
num_neutral = ds_russian['train'].filter(lambda row: row['label'] == 1).num_rows
num_positive = ds_russian['train'].filter(lambda row: row['label'] == 2).num_rows

In [14]:
print('Number of negative reviews: {}'.format(num_negative))
print('Number of neutral reviews: {}'.format(num_neutral))
print('Number of positive reviews: {}'.format(num_positive))

Number of negative reviews: 3788
Number of neutral reviews: 3227
Number of positive reviews: 104652


In [15]:
df_russian_negative = ds_russian['train'].filter(lambda row: row['label'] == 0).shuffle(seed=42).select(range(num_negative))
df_russian_neutral = ds_russian['train'].filter(lambda row: row['label'] == 1).shuffle(seed=42).select(range(num_neutral))
df_russian_positive = ds_russian['train'].filter(lambda row: row['label'] == 2).shuffle(seed=42).select(range(num_negative)) #YES, we reduce numbers

In [16]:
df_russian_reduced = concatenate_datasets([df_russian_negative, df_russian_neutral, df_russian_positive]).shuffle(seed=42)

In [17]:
df_russian_reduced

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label'],
    num_rows: 10803
})

In [18]:
ds_russian_splitted = df_russian_reduced.train_test_split(test_size=0.1, seed=42)

In [19]:
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}

In [20]:
def get_system_prompt(negative_example, neutral_example, positive_example):
    return f"""
Твоя задача определить тональность отзыва из маркетплейса.

Отзыв будет обернут в xml тэги <review> </review>

Пример негативного отзыва:
<review>
{negative_example}
</review>

Пример нейтрального отзыва:
<review>
{neutral_example}
</review>

Пример позитивного отзыва:
<review>
{positive_example}
</review>

Отвечай только одной цифрой с номером события

- Негативный отзыв: 0
- Нейтральный отзыв: 1
- Позитивный отзыв: 2
"""

In [21]:
def get_user_prompt(review):
    return f"""
Определи тональность следующего текста. Ответ дай одной цифрой.
<review>{review}</review>
"""

In [22]:
llm = ChatOpenAI(
    base_url=os.getenv('BASEURL'),
    model='qwen-3-32b',
    temperature=0.0,
    api_key=os.getenv('APIKEY'))

In [23]:
system_prompt = SystemMessagePromptTemplate.from_template(
    get_system_prompt(df_russian_negative[0]['combined_text'],
                      df_russian_neutral[0]['combined_text'],
                      df_russian_positive[0]['combined_text'])
)
print(system_prompt)

prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\nТвоя задача определить тональность отзыва из маркетплейса.\n\nОтзыв будет обернут в xml тэги <review> </review>\n\nПример негативного отзыва:\n<review>\nне знать ,  почему данный вещь один положительный отзыв ... допускать ,  конкретно не везти этот мясорубка . просто кошмар ! она не молоть мясо ,  она ,  хороший случай ,  выдавливать . максимум ,  она сила - этот куринный филе ,  булка лук ! она пытаться смалывать не самый плохой жесткий говядина ,  превращаться просто мука . давить такой сила ,  глядеть стол проломиться агрегат сломаться ... и ,  не давать бог ,  прийти голова вымывать разборный часть посудомойка - этот облезлый шероховатый ... ждать нетерпение ,  сдыхать ,  чистый совесть купить другой никакой ! легко вручную смалывать мясо ,  агрегат \n\n</review>\n\nПример нейтрального отзыва:\n<review>\nтвердый 2+ хороший твердый овощ ,  сок получаться не чистый мякоть ,  приходиться проце

In [24]:
def predict_sentiment(row):
    user_prompt = HumanMessage(get_user_prompt(row['combined_text']))
    llm_resp = (ChatPromptTemplate.from_messages([system_prompt, user_prompt]) | llm | StrOutputParser()).invoke({})
    return {
        'predicted': int(llm_resp[0])
    }

In [25]:
ds_russian_splitted_with_prediction = ds_russian_splitted['test'].map(predict_sentiment)

Parameter 'function'=<function predict_sentiment at 0x7f64b0a59120> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/1081 [00:00<?, ? examples/s]

In [26]:
ds_russian_splitted_with_prediction

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label', 'predicted'],
    num_rows: 1081
})

In [28]:
print(classification_report(ds_russian_splitted_with_prediction["label"], ds_russian_splitted_with_prediction["predicted"], target_names=[id2label[i] for i in range(NUM_LABELS)]))

              precision    recall  f1-score   support

    negative       0.71      0.84      0.77       395
     neutral       0.63      0.45      0.53       319
    positive       0.85      0.89      0.87       367

    accuracy                           0.74      1081
   macro avg       0.73      0.73      0.72      1081
weighted avg       0.73      0.74      0.73      1081

